# Playground Series S6E8: Predicting Smartphone Addiction | S6E8 | Smartphone Addiction Ceiling Breaker 解説付き写し

- **コンペ**: [Predicting Smartphone Addiction](https://www.kaggle.com/competitions/playground-series-s6e8)（Playground Series - Season 6 Episode 8。合成データを使ったスマートフォン依存度の予測タスク）
- **元notebook**: [S6E8 | Smartphone Addiction Ceiling Breaker](https://www.kaggle.com/code/anhadmahajan06/s6e8-smartphone-addiction-ceiling-breaker)（ANHAD MAHAJAN氏、Public/Best Score **0.97090**）
- **手法の概要**: このnotebookは自分でモデルを学習させるものではなく、**他の公開notebookが出力した「予測結果CSV」を複数集めてきて、賢くブレンド（合成）する**ことでスコアを底上げする「アンサンブル特化」notebookです。特に、単純な平均では伸びない状況（＝複数の予測がほぼ同じ内容で相関しすぎている状況）を見抜き、**ランク変換＋非線形変換（logit空間でのブレンド、べき乗による強調など）**を使って予測の「差」を強調する4種類のブレンド戦略を出力します。
- **スコア**: Public/Best Score **0.97090**

> これは学習目的の解説付き写しです。原著者のコード自体は改変していませんが、出力（実行結果・ログ）は含まれていません（未実行）。

## 評価指標

- **タスク**: 合成データ上の各サンプルについて、「スマートフォン依存」に該当するかどうかを2値（0/1）で予測する2値分類タスク（`addicted_label`列）。
- **評価指標**: このコンペは **ROC-AUC（Receiver Operating Characteristic - Area Under Curve）** で評価されます。ROC-AUCは「陽性サンプルと陰性サンプルをランダムに1つずつ選んだとき、モデルが陽性サンプルに高いスコアを付ける確率」を表す指標で、0.5がランダム予測、1.0が完全な予測です。**予測値の絶対的な大きさではなく、サンプル間の並び順（ランク）だけが評価に影響する**という特徴があります。
- **なぜこの指標が使われるか**: 依存/非依存のようなクラス分類タスクでは、しきい値（例えば0.5)を固定した正解率よりも、ROC-AUCの方が「モデルがどれだけ陽性と陰性を正しく順序付けられているか」を閾値に依存せず評価でき、クラスの偏り（不均衡）にもある程度頑健です。
- **このnotebookの手法とROC-AUCの関係**: ROC-AUCが**順位（ランク）のみ**に依存するという性質を逆手にとって、このnotebookは予測値を`rankdata`で順位に変換してからブレンドしています。さらに、順位を[0,1]に正規化した後 `logit` 変換（0〜1の確率をオッズの対数に変換する関数）で「引き伸ばし」、極端に自信のある予測とそうでない予測の差を強調してからブレンドし直すことで、ほぼ同じ内容の予測同士を単純平均するより情報量を増やそうとしています。


## 1. 必要なライブラリの読み込み

**何をしているか**: `numpy`・`pandas`に加えて、`scipy.stats.rankdata`（値を順位に変換する関数）と`scipy.special.logit, expit`（確率とオッズの対数を相互変換する関数）を読み込んでいます。

**なぜそうするのか**: 後述のブレンド処理で「予測値をランクに変換する」「ランクをlogit空間に持ち上げて強調してから戻す」という操作を行うため、この2つの関数が中心的な道具になります。`logit(p) = log(p / (1-p))`で、pが0や1に近づくほど絶対値が大きくなる（＝「自信の強さ」を増幅する）性質を持ちます。`expit`はその逆変換（シグモイド関数）です。


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.special import logit, expit
from pathlib import Path
import glob
import re
import warnings

warnings.filterwarnings('ignore')


## 2. 提出ファイル（submission）が置かれているフォルダを自動検出

**何をしているか**: Kaggle Notebook環境の`/kaggle/input`以下を探索し、フォルダ名に`smartphone-addiction-submission`を含むディレクトリと、`playground-series`を含む`sample_submission.csv`を自動的に見つけます。

**なぜそうするのか**: Kaggle Datasetsとしてアタッチされたフォルダのパスは実行環境によって変わることがあるため、決め打ちのパスに依存せず自動探索することで、どの環境でも同じコードがそのまま動くようにしています（初心者がつまずきやすい「パスが違ってエラーになる」問題を避ける工夫）。


In [ ]:
def auto_detect_paths():
    paths = {'submission_dir': None, 'sample_sub': None}
    search_dirs = [Path('/kaggle/input'), Path('/input'), Path('.'), Path('..')]
    
    for base in search_dirs:
        if not base.exists(): continue
        for p in base.rglob('*'):
            if p.is_dir() and 'smartphone-addiction-submission' in p.name.lower():
                paths['submission_dir'] = p
        for p in base.rglob('sample_submission.csv'):
            if 'playground-series' in str(p).lower():
                paths['sample_sub'] = p
            
    if paths['submission_dir'] is None:
        paths['submission_dir'] = Path('./ps-s6e8predicting-smartphone-addiction-submission')
    if paths['sample_sub'] is None:
        paths['sample_sub'] = Path('./playground-series-s6e8/sample_submission.csv')
        
    return paths

paths = auto_detect_paths()
print(f"Submission Directory: {paths['submission_dir']}")


## 3. 複数の公開submissionを集めて「動的重み」を計算する

**何をしているか**: `/kaggle/input`以下を再帰的に探索し、ファイル名に`0.xxxx`のようなスコアらしき数値を含むCSVファイルをすべて集めます。集めた各ファイルのスコアから、**指数関数的な重み**（`norm_score ** 4.0`）を計算し、スコアが高いモデルほど極端に大きな発言権を持つようにします。

**なぜそうするのか**: 単純にスコアで線形に重み付けすると、僅差の1位と2位のモデルでもほぼ同じ重みになってしまいます。ここでは正規化したスコアを4乗することで、**最高スコアのモデルにほぼ重みを集中させつつ、他のモデルもタイブレーカー（僅かな差をつける役）として少しだけ効かせる**という設計にしています（さらに全モデルに最低1%の重みを足すことで、重みがゼロになるのを防いでいます）。この「べき乗で重みを尖らせる」考え方は、アンサンブル学習で「信頼できるモデルほど強く採用する」ときによく使われるテクニックです。


In [ ]:
# We search ALL directories in /kaggle/input for any CSVs containing a score in their name
search_base = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
sub_files = []
for p in search_base.rglob('*.csv'):
    if re.search(r"0\.[0-9]{4,}", p.name):
        sub_files.append(p)

print(f"Found {len(sub_files)} scored submission files across all directories.")

subs_dict = {}
for f in sub_files:
    fname = os.path.basename(f)
    match = re.search(r"0\.[0-9]{4,}", fname)
    score = float(match.group(0)) if match else 0.0
    
    if score > 0 and score not in subs_dict:  # Keep unique scores
        df = pd.read_csv(f).sort_values('id').reset_index(drop=True)
        subs_dict[score] = df['addicted_label'].values

# Sort dict from highest score to lowest
subs_dict = dict(sorted(subs_dict.items(), reverse=True))
scores = list(subs_dict.keys())
N = len(subs_dict[scores[0]])

# Dynamic Exponential Weights: Anchor top score, decay the rest
min_score = min(scores)
max_score = max(scores)

weights = []
for score in scores:
    if max_score == min_score:
        weights.append(1.0)
    else:
        # Scale between 0 and 1, then apply massive power to anchor to the top
        norm_score = (score - min_score) / (max_score - min_score)
        weights.append(norm_score ** 4.0)

# Add a tiny 1% base weight to prevent zeros
weights = [w + 0.01 for w in weights]

weight_sum = sum(weights)
weights = [w / weight_sum for w in weights]

print("--- Applied Dynamic Weights ---")
for s, w in zip(scores, weights):
    print(f"LB Score: {s:.5f} -> Weight: {w:.4f} ({w*100:.1f}%)")


## 4. 4種類の非線形ブレンド戦略で複数のsubmission.csvを出力

前提として、集めてきた複数のsubmission（予測）は「同じ公開notebookの派生」であることが多く、**互いの相関が非常に高い**（＝同じような間違い方をする）という問題があります。相関が高い予測同士を単純平均しても新しい情報は増えません。そこでこのセルでは、予測を非線形に変換してから重み付き平均する4つの戦略を試します。

**戦略1: Logit（対数オッズ）ブレンド（メイン提出）**
- **何をしているか**: 各予測をまずランクに変換して[0,1]に正規化し、それを`logit`変換してから重み付き平均し、最後に`expit`（シグモイド）で確率に戻します。
- **なぜそうするのか**: logit空間での平均は、確率空間での単純平均と異なり、「0.999と0.9999の違い」のような極端な値の微妙な差を引き伸ばして扱えます。ROC-AUCは順位だけで決まるため、この「裾（tail（を引き伸ばす」操作がサンプルの並び替えに影響し、スコア向上につながることがあります。

**戦略2: 極端アンカーブレンド（最高スコアモデルに90%の重み）**
- **何をしているか**: 最もスコアの高いモデルの予測に90%、残り全モデルの合計に10%の重みを与えて平均します。
- **なぜそうするのか**: 「一番信頼できるモデルの意見をほぼそのまま使いつつ、僅差の判定だけ他のモデルに"タイブレーク"してもらう」という考え方です。

**戦略3: シャープネス（べき乗）ブレンド**
- **何をしているか**: 各予測のランクを3乗してから重み付き平均します。
- **なぜそうするのか**: べき乗（0〜1の値を3乗する）は0.5付近の「自信がない」予測をさらに0や1に近づけて、予測全体をよりはっきり（シャープ）させる効果があります。「迷っている予測」を罰する形になります。

**戦略4: Min-Maxブレンド**
- **何をしているか**: 各予測を元のスケールのまま[0,1]にMin-Max正規化してから重み付き平均します。
- **なぜそうするのか**: ランク変換やlogit変換をせず、元の確率分布の形をできるだけ保ったまま複数モデルを混ぜる、比較的保守的な方法です。他の3つの「攻めた」戦略との比較対象（ベースライン的な役割）になるます。

4つのファイルをこの順（logit → 極端アンカー → シャープネス → Min-Max）で提出してスコアの伸びを確認する、という運用を想定しています。


In [ ]:
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
sample_df = pd.read_csv(paths['sample_sub']).sort_values('id').reset_index(drop=True)

def save_sub(preds, filename):
    sub = sample_df.copy()
    sub['addicted_label'] = preds
    sub.to_csv(OUTPUT_DIR / filename, index=False)
    print(f"Saved: {filename}")

# ==============================================================
# STRATEGY 1: Log-Odds (Logit) Blending (THE MAIN SUBMISSION)
# Logit transforms stretch out the extreme predictions (the tails).
# This is hyper-effective for ROC-AUC because it separates the 0.999s from the 0.9999s.
# ==============================================================
logit_blend = np.zeros(N)
for i, (score, preds) in enumerate(subs_dict.items()):
    clipped_ranks = np.clip(rankdata(preds) / N, 1e-15, 1 - 1e-15)
    logit_blend += logit(clipped_ranks) * weights[i]
save_sub(expit(logit_blend), 'submission.csv')

# ==============================================================
# STRATEGY 2: Extreme Anchor Blending (90% Best / 10% Rest)
# Gives 90% of the voting power to the highest scoring model, using the others purely as tie-breakers.
# ==============================================================
anchor_blend = np.zeros(N)
anchor_blend += (rankdata(subs_dict[scores[0]]) / N) * 0.90
rest_weight = 0.10 / (len(scores) - 1)
for i in range(1, len(scores)):
    anchor_blend += (rankdata(subs_dict[scores[i]]) / N) * rest_weight
save_sub(anchor_blend, '2_submission_extreme_anchor.csv')

# ==============================================================
# STRATEGY 3: Sharpness Power Blending (Exponent 3.0)
# Punishes "unsure" predictions (near 0.5) and forces the distribution outwards.
# ==============================================================
power_blend = np.zeros(N)
for i, (score, preds) in enumerate(subs_dict.items()):
    power_blend += ((rankdata(preds) / N) ** 3.0) * weights[i]
save_sub(power_blend, '3_submission_power_sharp.csv')

# ==============================================================
# STRATEGY 4: Min-Max Normalized Blending
# Retains the original uncalibrated probability shapes but normalizes them to exactly [0,1].
# ==============================================================
minmax_blend = np.zeros(N)
for i, (score, preds) in enumerate(subs_dict.items()):
    min_val, max_val = np.min(preds), np.max(preds)
    normalized = (preds - min_val) / (max_val - min_val)
    minmax_blend += normalized * weights[i]
save_sub(minmax_blend, '4_submission_minmax_blend.csv')

print("\n All 4 Non-Linear Breakout Strategies Generated! Submit them starting with #1.")
